In [ ]:
from importlib import reload
from keras.saving import load_model
from keras.utils import load_img
from PIL import Image
import cv2
import geopandas as gpd
import numpy as np
import os
import pandas as pd
import rasterio
from rasterio.features import rasterize
import seaborn as sns
import segmenteverygrain as seg
import sez
from shapely import wkt
from shapely.geometry import Polygon
from skimage.measure import regionprops, regionprops_table
import tifffile
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from segment_anything import SamPredictor, sam_model_registry

# Extracting morphometric information from segmented grains
### 
- this notebook is intended to be run after all grains have been segmented, using either the create_polygons_and_masks.ipynb file or otherwise

### Step 1: Read in the original image

In [ ]:
original_image_path = '/Users/omw339/Desktop/OWBP25012/OWBP25012/OWBP25012.tif'
original_image = cv2.imread('/Users/omw339/Desktop/OWBP25012/OWBP25012/OWBP25012.tif')

### Step 2: Read in the Csv file with the coordinates of the segmented polygon to re-initalize the geodataframe

In [ ]:
# Replace with path to your csv file of the segmented polygon coordinates
csv_path = '/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_coordinates_9_16_25_final.csv'

In [ ]:
height, width = original_image.shape[:2]

# Create shapes generator: (geometry, label) tuples
gdf = sez.load_polygons("path/to/your/file.csv", crs="EPSG:4326")
shapes = ((geom, idx + 1) for idx, geom in enumerate(gdf.geometry))

# Rasterize polygons:
label_image = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    fill=0,
    dtype=np.uint16
)
all_grains = list(gdf.geometry)
print(f"Rasterized label image shape: {label_image.shape}")
print(f"Max label value (should equal number of polygons): {label_image.max()}")

Ensure that the max label value is equivalent to the number of grains you segmented, otherwise double-check your csv file

In [ ]:
tifffile.imwrite('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25012/OWBP25012_draft_10_04_rasterized_labels.tif', label_image)

### Step 3: Creating the morphometrics dataframe.
- The following cell creates the dataframe for the morphometric information. It is important to note that the original columns are in **pixels**

In [ ]:
from skimage.measure import regionprops_table

# Define the properties you want
properties = [
    'label',
    'area',
    'centroid',
    'bbox',
    'major_axis_length',
    'minor_axis_length',
    'eccentricity',
    'solidity',
    'orientation',
    'perimeter'
]

# Extract properties for each labeled region (grain)
props = regionprops_table(label_image, properties=properties)

# Convert to DataFrame
grain_data = pd.DataFrame(props)

print(grain_data.head())


### Step 4: The following cells are used to get morphometric information in micrometers (or any other unit)

**Option 1**: Use the length of the scale bar in pixels to get the scale of the image (in units / pixel) within the notebook. Run this cell and then click (left mouse button) on one end of the scale bar in the image and click (right mouse button) on the other end of the scale bar:

In [ ]:
image = np.array(load_img(original_image_path))
fig, ax = plt.subplots()
sez.plot_grains(original_image_path, all_grains, step='Deletions')

In [ ]:
cid5 = fig.canvas.mpl_connect('button_press_event', lambda event: seg.click_for_scale(event, ax))

**Option 2 (Recommended)**: Open up the original image in any image processing software such as FIJI or ImageJ to measure the scalebar

- n_of_units: represents the number on the scale bar in the image
- scale_bar_length: the length of the scale bar in pixels that you measured


In [ ]:
n_of_units = 1000 # micrometers usually'
scale_bar_length = 1802.167 #length of scale bar in pixels
units_per_pixel = n_of_units/scale_bar_length

Adding columns to the grain_data dataframe for the morphometric measurements in the scaled measurement of your choice

In [ ]:
grain_data_micron = sez.convert_grain_units(grain_data, units_per_pixel)

Adding measurements like roundness and aspect ratio. Now that you have the grain geometries, you can derive any other morphometric parameters as you see fit.

In [ ]:
grain_data['roundness'] = (4 * grain_data['area']) / (np.pi * (grain_data['major_axis_length']**2))
grain_data['aspect_ratio'] = grain_data['major_axis_length'] / grain_data['minor_axis_length']

In [ ]:
print(grain_data)

saving morphometric dataframe to csv

In [ ]:
grain_data.to_csv('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_9_16_morphometrics.csv')

## Optional Visualizations

Figure 1: Scatterplot of Aspect Ratios 

In [ ]:
x = grain_data['minor_axis_length_micron']
y = grain_data['major_axis_length_micron']

plt.scatter(x, y, c = grain_data['area_micron2'], cmap='gnuplot', s=50)

plt.xlabel('Minor Axis Length (mm)', fontsize=14)
plt.ylabel('Major Axis Length (mm)', fontsize=14)
plt.title('Zircon Axes Lengths', fontsize = 20)
plt.colorbar(label = 'Area (mm)')

plt.show()

Figure 2: Histogram of Aspect Ratios

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(grain_data['aspect_ratio'].dropna(), bins=30, kde=False, color='lightblue', edgecolor='black')
plt.title("Sample 2 Aspect Ratios")
plt.xlabel("Aspect Ratio (µm)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Inspecting a Particular Grain (Outlier Verification)
The following cells to extract the morphometric data and shape for a particular grain ID to confirm accuracy of outliers.

In [ ]:
chosen_grain = input("Please enter a grain number from 1 to " + str(len(gdf)) + ":")
gnumupdated = chosen_grain
gnumindex = int(chosen_grain) - 1

while len(gnumupdated) !=4:
    gnumupdated = "0" + gnumupdated

Once you have selected a grain_id, the following cell prints out its associated morphometric data

In [ ]:
grain_data_specific = grain_data[grain_data["label"] == int(chosen_grain)]
print("Here is your grain's data:")
grain_data_specific

This cell produces an overlay of the segmentation (in a thin green line) over the grain from the original image

In [ ]:
sez.show_grain_overlay(chosen_grain, grain_data, label_image, original_image)